### Dataset Load

In [10]:
from datasets import load_dataset

dataset = load_dataset("anindya64/hardhat")
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'objects'],
        num_rows: 5297
    })
    test: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'objects'],
        num_rows: 1766
    })
})

In [11]:
#### Each Datapoint looks like this

dataset["train"][0]

# extract out the train and test set
train_dataset = dataset["train"]
test_dataset = dataset["test"]


### Load Image Processor
* 모델 파인튜닝을 하기전에, 사전학습에 사용되었던 학습법과 완전하게 동일한 방식으로 데이터 전처리를 진행해야 합니다.
* 전처리에 사용할 이미지 프로세서를 로드합니다.
* AutoImageProcessor를 사용하여도 되고, 특정 모델의 경우에는 다른 ImageProcessor를 사용해야합니다. (Owlv2 모델은 AutoImageProcessor 호환 모델)


In [12]:
from transformers import AutoImageProcessor

checkpoint = "google/owlv2-base-patch16-ensemble"

image_processor = AutoImageProcessor.from_pretrained(checkpoint)

### Preprocessing the Dataset
augmentation - albumentations 라이브러리를 사용하여 resizing, rotationg 등 학습데이터셋 랜덤 변환

In [13]:
import albumentations
import numpy as np
import torch

transform = albumentations.Compose(
    [
        albumentations.Resize(480, 480),
        albumentations.HorizontalFlip(p=1.0),
        albumentations.RandomBrightnessContrast(p=1.0),
    ],
    bbox_params=albumentations.BboxParams(format="coco", label_fields=["category"]),
)

image_processor가 요구하는 format 생성

In [14]:
def formatted_anns(image_id, category, area, bbox):
    annotations = []
    for i in range(0, len(category)):
        new_ann = {
            "image_id": image_id,
            "category_id": category[i],
            "isCrowd": 0,
            "area": area[i],
            "bbox": list(bbox[i]),
        }
        annotations.append(new_ann)

    return annotations

augmentation과 formatting을 적용한 image_processor 반환

In [15]:
# transforming a batch


def transform_aug_ann(examples):
    image_ids = examples["image_id"]
    images, bboxes, area, categories = [], [], [], []
    for image, objects in zip(examples["image"], examples["objects"]):
        image = np.array(image.convert("RGB"))[:, :, ::-1]
        out = transform(image=image, bboxes=objects["bbox"], category=objects["id"])

        area.append(objects["area"])
        images.append(out["image"])
        bboxes.append(out["bboxes"])
        categories.append(out["category"])

    targets = [
        {"image_id": id_, "annotations": formatted_anns(id_, cat_, ar_, box_)}
        for id_, cat_, ar_, box_ in zip(image_ids, categories, area, bboxes)
    ]

    return image_processor(images=images, annotations=targets, return_tensors="pt")

In [16]:
# Apply transformations for both train and test dataset

train_dataset_transformed = train_dataset.with_transform(transform_aug_ann)
test_dataset_transformed = test_dataset.with_transform(transform_aug_ann)

In [17]:
train_dataset_transformed[0]

/Users/dhk/.local/share/virtualenvs/ml-workflow-8AcGETWo/lib/python3.10/site-packages/transformers/image_processing_utils.py:41: UserWarning: The following named arguments are not valid for `Owlv2ImageProcessor.preprocess` and were ignored: 'annotations'
  return self.preprocess(images, **kwargs)


{'pixel_values': tensor([[[ 0.0170,  0.0170,  0.0152,  ..., -0.2640, -0.4118, -0.4118],
          [ 0.0170,  0.0170,  0.0152,  ..., -0.2640, -0.4118, -0.4118],
          [ 0.0152,  0.0152,  0.0097,  ..., -0.2731, -0.4246, -0.4246],
          ...,
          [-0.4319, -0.4319, -0.5432,  ..., -0.6919, -0.7467, -0.7467],
          [-0.4337, -0.4337, -0.5487,  ..., -0.7175, -0.7795, -0.7795],
          [-0.4337, -0.4337, -0.5487,  ..., -0.7175, -0.7795, -0.7795]],
 
         [[ 0.0329,  0.0329,  0.0310,  ...,  0.1558, -0.0056, -0.0056],
          [ 0.0329,  0.0329,  0.0310,  ...,  0.1558, -0.0056, -0.0056],
          [ 0.0310,  0.0310,  0.0254,  ...,  0.1370, -0.0318, -0.0318],
          ...,
          [-0.0196, -0.0196, -0.1566,  ..., -0.4108, -0.4671, -0.4671],
          [-0.0215, -0.0215, -0.1622,  ..., -0.4370, -0.5008, -0.5008],
          [-0.0215, -0.0215, -0.1622,  ..., -0.4370, -0.5008, -0.5008]],
 
         [[ 0.2964,  0.2964,  0.2946,  ...,  0.7683,  0.6155,  0.6155],
          [ 

#### Collate Preprocessing

* 데이터 세트에서 샘플 목록을 가져와 모델의 입력 형식에 적합한 배치로 변환하는 역할 (패딩, 잘라내기) 수행.
* 데이터를 어떻게 batch 처리로 그룹화 할지 또는 각 batch 처리를 어떻게 표현할지 정의.
* `data collator`는 주로 데이터를 모아서 사전 처리함.

In [18]:
def collate_fn(batch):
    pixel_values = [item["pixel_values"] for item in batch]
    encoding = image_processor.pad(pixel_values, return_tensors="pt")
    labels = [item["labels"] for item in batch]
    batch = {}
    batch["pixel_values"] = encoding["pixel_values"]
    batch["pixel_mask"] = encoding["pixel_mask"]
    batch["labels"] = labels
    return batch

### Training

1. 기반 사전학습 model을 AutoModelForObjectDetection(owlv2의 경우 Owlv2ForObjectDetection)로 로드
2. TrainingArguments에 모든 hyperparameters와 추가 arguments 정의
3. Huggingface Trainer에 학습 arguments 및 Model , ImageDataset 주입.
4. train() 실행

In [19]:
from transformers import AutoModelForObjectDetection, Owlv2ForObjectDetection

id2label = {0: "head", 1: "helmet", 2: "person"}
label2id = {v: k for k, v in id2label.items()}


model = Owlv2ForObjectDetection.from_pretrained(
    checkpoint,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

In [24]:
from transformers import TrainingArguments
from transformers import Trainer

# Define the training arguments

training_args = TrainingArguments(
    output_dir="models/owlv2-finetuned",
    per_device_train_batch_size=8,
    num_train_epochs=1,
    max_steps=1000,
    # fp16=True,
    save_steps=10,
    logging_steps=30,
    learning_rate=1e-5,
    weight_decay=1e-4,
    save_total_limit=2,
    remove_unused_columns=False,
)

# Define the trainer

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_dataset_transformed,
    eval_dataset=test_dataset_transformed,
    tokenizer=image_processor,
)

trainer.train()

ValueError: fp16 mixed precision requires a GPU (not 'mps').